Importing the necessary functions and loading the model 

In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


In [2]:
import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

In [ ]:
import importlib 
import eval_utils
importlib.reload(eval_utils)
from eval_utils import load_model
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-27/xrd_no_encoder_atomic_comp_concat_and_mask_natoms_given_mp_20_v3")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/9046 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/

In [154]:
model.to('cuda:0')

CDVAE(
  (encoder): DimeNetPlusPlusWrap(
    (rbf): BesselBasisLayer(
      (envelope): Envelope()
    )
    (sbf): SphericalBasisLayer(
      (envelope): Envelope()
    )
    (emb): EmbeddingBlock(
      (emb): Embedding(95, 128)
      (lin_rbf): Linear(in_features=6, out_features=128, bias=True)
      (lin): Linear(in_features=384, out_features=128, bias=True)
    )
    (output_blocks): ModuleList(
      (0): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin_up): Linear(in_features=128, out_features=256, bias=True)
        (lins): ModuleList(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Linear(in_features=256, out_features=256, bias=True)
          (2): Linear(in_features=256, out_features=256, bias=True)
        )
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
      (1): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin

In [156]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35


In [157]:
def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

In [158]:
batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

In [159]:
batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

In [160]:
_, _, z = model.encode(batch, xrd_int, xrd_loc, atom_spec)

In [161]:
#get the first true crystal structure 
print(batch)

Batch(edge_index=[2, 19564], y=[256, 1], frac_coords=[2651, 3], atom_types=[2651], lengths=[256, 3], angles=[256, 3], to_jimages=[19564, 3], num_atoms=[256], num_bonds=[256], num_nodes=2651, batch=[2651], ptr=[257])


First stage: show that the code works for a single crystal with stripped-down code 

In [162]:
num_atoms = batch.num_atoms[[0]].to('cuda:0')
atom_types = atom_spec[[0]].to('cuda:0')
z = z[[0]].to('cuda:0')

In [163]:
print(num_atoms)
print(atom_types)
print(z)

tensor([8], device='cuda:0')
tensor([[31, 52,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,

In [164]:
#### inputs 
gt_num_atoms = num_atoms
gt_elements_input = atom_types
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
            z, gt_num_atoms, gt_elements=gt_elements_input)
composition_per_atom = composition_per_atom.to('cuda:0') # this can be removed in native code 
num_atoms = batch.num_atoms[[0]].to('cuda:0')  #this can be removed in native code 

# Create a range tensor and repeat it as before
range_tensor = torch.arange(len(num_atoms), device='cuda:0')
crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

# Convert atom_types into a mask
atom_mask = atom_types != 0

# For each unique crystal_id, get its corresponding indices in composition_per_atom
unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)

composition_per_atom = composition_per_atom + 1

start_idx = 0
for u_id, count in zip(unique_crystal_ids, counts):
    relevant_elements = atom_types[u_id][atom_mask[u_id]]
    mask = torch.ones_like(composition_per_atom[start_idx])
    mask *= (-10**6)
    mask[relevant_elements-1] = 1

    # Create additive mask
    additive_mask_for_normalization = mask 
    additive_mask_for_normalization[relevant_elements-1] *= 0.0001

    # Apply masks to the relevant segment of composition_per_atom
    composition_per_atom[start_idx:start_idx+count] *= mask
    composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

    # Update start index for next iteration
    start_idx += count

In [165]:
self = model

In [166]:
batch.num_atoms = batch.num_atoms.to('cuda:0')

In [167]:
noise_level = torch.randint(0, self.sigmas.size(0),
                                    (batch.num_atoms.size(0),),
                                    device=self.device)
used_sigmas_per_atom = self.sigmas[noise_level].repeat_interleave(
    batch.num_atoms, dim=0)

type_noise_level = torch.randint(0, self.type_sigmas.size(0),
                                (batch.num_atoms.size(0),),
                                device=self.device)
used_type_sigmas_per_atom = (
    self.type_sigmas[type_noise_level].repeat_interleave(
        batch.num_atoms, dim=0))

In [168]:
pred_composition_per_atom = composition_per_atom
pred_composition_probs = F.softmax(
            pred_composition_per_atom.detach(), dim=-1)

In [169]:
print(pred_composition_probs)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0

In [170]:
batch.atom_types = batch.atom_types.to('cuda:0')

In [171]:
atom_type_probs = (
    F.one_hot(batch.atom_types[[range(8)]] - 1, num_classes=MAX_ATOMIC_NUM) +
    pred_composition_probs)

In [172]:
rand_atom_types = torch.multinomial(
    atom_type_probs, num_samples=1).squeeze(1) + 1

In [173]:
rand_atom_types

tensor([31, 31, 31, 31, 52, 52, 31, 31], device='cuda:0')

In [174]:
batch.frac_coords = batch.frac_coords.to('cuda:0')
batch.num_atoms = batch.num_atoms.to('cuda:0')
batch.lengths = batch.lengths.to('cuda:0')
batch.angles = batch.angles.to('cuda:0')

In [175]:
(pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = self.decode_stats(
            z, batch.num_atoms[[0]], batch.lengths[[0]], batch.angles[[0]], True, gt_elements)

In [176]:
cart_noises_per_atom = (
    torch.randn_like(batch.frac_coords[[8]]))
cart_coords = frac_to_cart_coords(
    batch.frac_coords[[8]], pred_lengths[[0]], pred_angles[[0]], batch.num_atoms[[0]])
cart_coords = cart_coords + cart_noises_per_atom
noisy_frac_coords = cart_to_frac_coords(
    cart_coords[[0]], pred_lengths[[0]], pred_angles[[0]], batch.num_atoms[[0]])

In [177]:
pred_cart_coord_diff, pred_atom_types = self.decoder(
    z, noisy_frac_coords[[0]], rand_atom_types[[range(8)]], batch.num_atoms[[0]], pred_lengths[[0]], pred_angles[[0]], gt_elements[[0]])

In [178]:
print(pred_atom_types)

tensor([[  0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,  -7.0912,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,  -7.7928,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0

The resultant vector should have extremely large negative numbers in all places where the value doesn't correspond to the correct elements 

Second stage: creating a function and verifying the function 

In [179]:
def composition_constraint(self, atom_types, num_atoms, composition_per_atom):
    """
    Restrict the probability distribution from which the atom types are randomly drawn 
    to only include the elements that are present in the crystal.

    atom_types: the atom types in the crystal
    num_atoms: the number of atoms in the crystal
    composition_per_atom: the probability distribution from which the atom types are randomly drawn 

    """

    # Create a range tensor and repeat it as before
    range_tensor = torch.arange(len(num_atoms), device='cuda:0')
    crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

    # Convert atom_types into a mask
    atom_mask = atom_types != 0

    # For each unique crystal_id, get its corresponding indices in composition_per_atom
    unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)

    composition_per_atom = composition_per_atom + 1

    start_idx = 0
    for u_id, count in zip(unique_crystal_ids, counts):
        relevant_elements = atom_types[u_id][atom_mask[u_id]]
        mask = torch.ones_like(composition_per_atom[start_idx])
        mask *= (-10**6)
        mask[relevant_elements-1] = 1

        # Create additive mask
        additive_mask_for_normalization = mask 
        additive_mask_for_normalization[relevant_elements-1] *= 0.0001

        # Apply masks to the relevant segment of composition_per_atom
        composition_per_atom[start_idx:start_idx+count] *= mask
        composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

        # Update start index for next iteration
        start_idx += count
    
    return composition_per_atom

In [180]:
gt_num_atoms = num_atoms
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
            z, gt_num_atoms, gt_elements=gt_elements_input)
composition_per_atom = composition_per_atom.to('cuda:0') # this can be removed in native code 
num_atoms = batch.num_atoms[[0]].to('cuda:0')  #this can be removed in native code 

composition_constraint(model, atom_types, num_atoms, composition_per_atom)

tensor([[-2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
          3.6399e-04, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06,  3.2929e-04, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.

Third stage: ensuring that the function works in all contexts in which it is applied. 

Because I validated the performance of the code in a workflow modeled after the forward pass, and the other situations in which the code would be used (langevin dynamics) are highly similar to the forward pass (and I would modularize it into the decode states and decode functions themselves), I don't think further notebook code is neccesary for this third stage in this particular case. 

However, I will take the following steps to verify performance "in-vivo": 
1. For the first few steps / epochs, I will have the code print out the results from the composition constraint ("before/after") that kind of thing. I will do this in the decoder stats function and before and after the decode call at the end of the forward pass. For the langevin dynamics, I will just print it out for every batch because there's less of a risk of slowing things down or taking up memory. After verifying, I'll save the outputs for posterity and get rid of the print statements. 